# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

[MOCK] ScrapedDeal.fetch called - returning mock scraped deals


In [3]:
len(deals)

Out[1]: 2


In [4]:
deals[10].describe()

---------------------------------------------------------------------------
IndexError                                Traceback (most recent call last)
Cell In[1], line 1
----> 1 deals[10].describe()

IndexError: list index out of range


### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Hisense TV
Details: 55-inch Roku Smart TV
Features: 4K UHD

Title: Poly Studio P21
Details: 21.5-inch meeting display

Include exactly 5 deals, no more.


In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

[MOCK] OpenAI parse called for format <class 'agents.deals.DealSelection'>
Out[1]: DealSelection(deals=[Deal(product_description='HyperX QuadCast S USB Condenser Microphone with RGB lighting', price=119.0, url='https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39'), Deal(product_description='Sony WH-1000XM4 Wireless Noise Canceling Over-Ear Headphones', price=228.0, url='https://dealnews.com/item2'), Deal(product_description='Logitech MX Master 3S Wireless Performance Mouse', price=89.0, url='https://dealnews.com/item3'), Deal(product_description='Apple iPad Air 10.9-inch 64GB Wi-Fi (Latest Model)', price=499.0, url='https://dealnews.com/item4'), Deal(product_description='Nintendo Switch OLED Model with White Joy-Con', price=309.0, url='https://dealnews.com/item5')])


In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


HyperX QuadCast S USB Condenser Microphone with RGB lighting
119.0
https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39

Sony WH-1000XM4 Wireless Noise Canceling Over-Ear Headphones
228.0
https://dealnews.com/item2

Logitech MX Master 3S Wireless Performance Mouse
89.0
https://dealnews.com/item3

Apple iPad Air 10.9-inch 64GB Wi-Fi (Latest Model)
499.0
https://dealnews.com/item4

Nintendo Switch OLED Model with White Joy-Con
309.0
https://dealnews.com/item5



In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

[MOCK] ScrapedDeal.fetch called - returning mock scraped deals
[MOCK] OpenAI parse called for format <class 'agents.deals.DealSelection'>


INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 2 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

Out[1]: DealSelection(deals=[Deal(product_description='HyperX QuadCast S USB Condenser Microphone with RGB lighting', price=119.0, url='https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39'), Deal(product_description='Sony WH-1000XM4 Wireless Noise Canceling Over-Ear Headphones', price=228.0, url='https://dealnews.com/item2'), Deal(product_description='Logitech MX Master 3S Wireless Performance Mouse', price=89.0, url='https://dealnews.com/item3'), Deal(product_description='Apple iPad Air 10.9-inch 64GB Wi-Fi (Latest Model)', price=499.0, url='https://dealnews.com/item4'), Deal(product_description='Nintendo Switch OLED Model with White Joy-Con', price=309.0, url='https://dealnews.com/item5')])


### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [14]:
load_dotenv(override=True)

Out[1]: True


In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user not found
Pushover token not found


In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!
[MOCK] Pushover message sent: MASSIVE DEAL!!


In [19]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

[MOCK] Pushover message sent: SUCH A MASSIVE DEAL!!


In [20]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

---------------------------------------------------------------------------
AuthenticationError                       Traceback (most recent call last)
Cell In[1], line 1
----> 1 agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

File c:\Users\shobhasreenivas\projects\llm_engineering\week8\agents\messaging_agent.py:70, in MessagingAgent.notify(self, description, deal_price, estimated_true_value, url)
     66 """
     67 Make an alert about the specified details
     68 """
     69 self.log("Messaging Agent is using Claude to craft the message")
---> 70 text = self.craft_message(description, deal_price, estimated_true_value)
     71 self.push(text[:200] + "... " + url)
     72 self.log("Messaging Agent has completed")

File c:\Users\shobhasreenivas\projects\llm_engineering\week8\agents\messaging_agent.py:57, in MessagingAgent.craft_message(self, description, deal_price, estimated_true_value)
     55 user_prompt += f"Item Desc